# Annotating EMISSOR scenario

Annotations are added to the EMISSOR JSON file as interpretations of segments of a signal. In this notebook, we will only annotate text signals. Please consult the [Leolani Github](https://github.com/leolani) for annotations of other signals such as audio and images.

We will consider four annotations:

1. Dialogue acts
2. Emotions and sentiments
3. Tokens, their part-of-speech and named entities
4. The likelihood of utterances in a sequence

These annotations will give an impression of the quality of the interaction as communication.

Annoations are attached to signals as part of a so-called ```mention```. A mention is a data element that contains a specification of the segment from the signal that is annotated (in the case of text offset positions, in the case of an image bounding boxes) and the annoation itself.
Below is an example of a mention where we focus on an annotation element:

```
      {
        "@context": {...},
        "@type": "Mention",
        "segment": [
          {...}
        ],
        "annotations": [
          {
            "timestamp": 1732101081681,
            "@context": {...},
            "value": {
              "type": "GO",
              "value": "neutral",
              "confidence": 0.6464349627494812,
              "_py_type": "emissor.representation.util-JSON"
            },
            "@type": "Annotation",
            "source": "GO",
            "type": "python-type:cltl.emotion_extraction.api.Emotion",
            "_py_type": "emissor.representation.scenario-Annotation"
          },
        ],
        "id": "0116513d-dbec-4a5c-a54f-d29f7732ec7a"
      },
```

There can be multiple annotations for the same segment of the same signal.

In [6]:
### Importing the annotators from installed pip modules
from cltl.dialogue_act_classification.add_dialogue_acts_to_emissor import DialogueActAnnotator
from cltl.emotion_extraction.add_emotions_to_emissor import EmotionAnnotator
from cltl.nlp.add_nlp_to_emissor import NLPAnnotator
from cltl.dialogue_evaluation.add_likelihood_to_emissor import LikelihoodAnnotator

### Importing the emissor functions to load and save emissor JSON
from emissor.persistence import ScenarioStorage
from emissor.persistence.persistence import ScenarioController
from emissor.processing.api import SignalProcessor
from emissor.representation.scenario import Modality, Signal

In [7]:
EMISSOR="./emissor"
SCENARIO="7a201bea-3e2f-4dc2-ac36-a628ea4775cd"
SCENARIO="d5a6bc60-c19b-4c08-aee5-b4dd1c65c64d"

### Cleaning any existing annotations

Annotations are added every time you call an annotator. To avoid adding duplicate annotations, the next function removes annotations from a source.
You should call this function before you annotate an emissor JSON to avoid duplicates. Inspect the JSON file to find the source value.

In [8]:
def remove_annotations(signals:[Signal], annotation_source:str):
    for signal in signals:
        keep_mentions = []
        for mention in signal.mentions:
            clear=False
            for annotation in mention.annotations:
                if annotation.source and annotation.source== annotation_source:
                    clear=True
                    break
            if not clear:
                keep_mentions.append(mention)
        signal.mentions = keep_mentions

## Assigning GO Emotions to text signals

In [24]:
model="bhadresh-savani/bert-base-go-emotion"
model_name = "GO"
annotator = EmotionAnnotator(model=model, model_name=model_name)
scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

2024-11-21 15:39:28 -     INFO -       cltl.emotion_extraction.utterance_go_emotion_extractor - got [[{'label': 'admiration', 'score': 0.0014867145800963044}, {'label': 'amusement', 'score': 0.0022335301619023085}, {'label': 'anger', 'score': 0.01286606676876545}, {'label': 'annoyance', 'score': 0.015300208702683449}, {'label': 'approval', 'score': 0.010155470110476017}, {'label': 'caring', 'score': 0.0015890321228653193}, {'label': 'confusion', 'score': 0.04706558585166931}, {'label': 'curiosity', 'score': 0.11035915464162827}, {'label': 'desire', 'score': 0.002388907130807638}, {'label': 'disappointment', 'score': 0.0022112540900707245}, {'label': 'disapproval', 'score': 0.0024674038868397474}, {'label': 'disgust', 'score': 0.0020692977122962475}, {'label': 'embarrassment', 'score': 0.0012947828508913517}, {'label': 'excitement', 'score': 0.002869222778826952}, {'label': 'fear', 'score': 0.001005017664283514}, {'label': 'gratitude', 'score': 0.0005970300990156829}, {'label': 'grief',

### Assigning MIDAS dialogue acts to text signals

In [27]:
model= "../leolani_text_to_ekg/resources/midas-da-xlmroberta"
model_name="MIDAS"
annotator = DialogueActAnnotator(model=model, model_name=model_name)

scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

You are using a model of type xlm-roberta to instantiate a model of type roberta. This is not supported for all configurations of models and can yield errors.


### Assigning LLM likelihood scores to text signals

In [28]:
model= "../leolani_text_to_ekg/resources/usr-topicalchat-roberta_ft"
model_name="USR"
annotator = LikelihoodAnnotator(model=model, 
                                model_name=model_name, 
                                max_content=300, 
                                top_results=20)

scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

## Adding spaCy NLP annotations

Depending on the language of the communication, you need to install the proper lanuage model from spaCY.

For English this can be done as follows. Within the same virtual environment, call the next command from the command line in a terminal:

```python -m spacy download en_core_web_sm```



In [9]:
model = 'en_core_web_sm'
model_name ="NLP"
annotator = NLPAnnotator(model=model)
scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, text_signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

### Show annotations

In [10]:
scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
for signal in signals:
    mentions = signal.mentions
    for mention in mentions:
        annotations = mention.annotations
        for annotation in annotations:
            print(annotation.type, annotation.value)

ConversationalAgent Ai2Thor
python-type:cltl.emotion_extraction.api.Emotion Emotion(type=<EmotionType.GO: 1>, value='neutral', confidence=0.760793924331665)
python-type:cltl.emotion_extraction.api.Emotion Emotion(type=<EmotionType.EKMAN: 2>, value='neutral', confidence=0.760793924331665)
python-type:cltl.emotion_extraction.api.Emotion Emotion(type=<EmotionType.SENTIMENT: 4>, value='neutral', confidence=0.7761824033223093)
python-type:cltl.dialogue_act_classification.api.DialogueAct DialogueAct(type='MIDAS', value='command', confidence=3.7178430557250977)
Likelihood 0.3647939035935061
Token Token(text='Hi', pos=<POS.INTJ: 7>, segment=(0, 2))
Token Token(text='Piek', pos=<POS.INTJ: 7>, segment=(3, 7))
Token Token(text='.', pos=<POS.PUNCT: 13>, segment=(7, 8))
Token Token(text='Tell', pos=<POS.VERB: 17>, segment=(9, 13))
Token Token(text='me', pos=<POS.PRON: 11>, segment=(14, 16))
Token Token(text='what', pos=<POS.PRON: 11>, segment=(17, 21))
Token Token(text='to', pos=<POS.PART: 10>, seg

### END